# ODSB-16780: Postback Analysis for `com.netmarble.sololv`

Author: @haewon.yum <br>
**Date**: 2026-03-09  
**App**: Solo Leveling ARISE (Android, `com.netmarble.sololv`)  
**MMP**: AppsFlyer  
**Suspicious campaigns**: `Jl5NUKiajhIyKXQK`, `fISFGcvrsN7uXPoR`, `nqWMPgcAgXjZgEM8`

## Key Question

**Is the missing revenue limited to specific campaign-attributed data, or does it also affect unattributed postbacks?**

- If raw postbacks (attributed + unattributed) also show $0 revenue → **MMP-side issue** (AppsFlyer not sending revenue values)
- If raw postbacks have revenue but attributed data shows $0 → **attribution/processing issue** on Moloco side

## Tables
- **Attributed**: `moloco-ae-view.athena.fact_dsp_all` (campaign-attributed events)
- **All postbacks (attributed + unattributed)**: `focal-elf-631.df_accesslog.pb` (~1TB/day, use with care)

In [1]:
from google.cloud import bigquery
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

client = bigquery.Client(project='moloco-ods')

def run_query(query, label=''):
    """Run a BQ query and return DataFrame. Print row count."""
    try:
        df = client.query(query).result().to_dataframe()
        status = f'✅ {label}: {len(df)} rows' if len(df) > 0 else f'⚠️ {label}: 0 rows — check needed'
        print(status)
        return df
    except Exception as e:
        print(f'❌ {label}: {e}')
        return pd.DataFrame()

print("Setup complete.")

START_DATE = '2026-03-01'
END_DATE   = '2026-03-16'

Setup complete.


## 1. All Attributed Events — Name, Type, Revenue

Check all event types/names for `com.netmarble.sololv` in the attributed postback table, with revenue indicators.

In [39]:

df_events = run_query(f"""
SELECT
  cv.pb.event.name AS event_name,
  COUNT(*) AS event_count,
  COUNTIF(cv.revenue_usd.amount > 0) AS events_with_revenue,
  COUNTIF(cv.revenue_usd.amount = 0 OR cv.revenue_usd.amount IS NULL) AS events_without_revenue,
  ROUND(SUM(IFNULL(cv.revenue_usd.amount, 0)), 2) AS total_revenue_usd
FROM `focal-elf-631.prod_stream_view.cv`
WHERE cv.pb.app.bundle = 'com.netmarble.sololv'
  AND DATE(timestamp) BETWEEN '{START_DATE}' AND '{END_DATE}'
  AND cv.pb.moloco.attributed = true
GROUP BY 1
ORDER BY event_count DESC
""", '1. All Attributed Events (cv)')

if not df_events.empty:
    print(f"Total event names: {len(df_events)}")
    print(f"\n=== Events WITH revenue ===")
    display(df_events[df_events['total_revenue_usd'] > 0])
    print(f"\n=== All events (top 30 by count) ===")
    display(df_events.head(30))

✅ 1. All Attributed Events (cv): 45 rows
Total event names: 45

=== Events WITH revenue ===


,event_name,event_count,events_with_revenue,events_without_revenue,total_revenue_usd
33,revenue,5947,5377,570,37484.59



=== All events (top 30 by count) ===


,event_name,event_count,events_with_revenue,events_without_revenue,total_revenue_usd
0,af_app_opened,554784,0,554784,0.0
1,visit_shop,404323,0,404323,0.0
2,login,376123,0,376123,0.0
3,gacha_normal_x10,231048,0,231048,0.0
4,lvup_hunter_1st,207577,0,207577,0.0
5,enhance_artifact_1st,192414,0,192414,0.0
6,cdn_download_finished,182135,0,182135,0.0
7,lvup_weapon_1st,138874,0,138874,0.0
8,lvup_skill_1st,110539,0,110539,0.0
9,gacha_limited_x10,103913,0,103913,0.0


In [40]:
df_events

,event_name,event_count,events_with_revenue,events_without_revenue,total_revenue_usd
0,af_app_opened,554784,0,554784,0.00
1,visit_shop,404323,0,404323,0.00
2,login,376123,0,376123,0.00
3,gacha_normal_x10,231048,0,231048,0.00
4,lvup_hunter_1st,207577,0,207577,0.00
5,enhance_artifact_1st,192414,0,192414,0.00
6,cdn_download_finished,182135,0,182135,0.00
7,lvup_weapon_1st,138874,0,138874,0.00
8,lvup_skill_1st,110539,0,110539,0.00
9,gacha_limited_x10,103913,0,103913,0.00


## 2. Campaign-Level Attributed Revenue Events

Daily event count and revenue amount for revenue events, broken down by campaign.

In [41]:
df_campaign_rev = run_query(f"""
SELECT
  cv.pb.moloco.campaign_id AS campaign_id,
  DATE(timestamp) AS date,
  COUNT(*) AS revenue_events,
  COUNTIF(cv.revenue_usd.amount > 0) AS events_with_revenue,
  ROUND(SUM(IFNULL(cv.revenue_usd.amount, 0)), 4) AS pb_revenue
FROM `focal-elf-631.prod_stream_view.cv`
WHERE cv.pb.app.bundle = 'com.netmarble.sololv'
  AND DATE(timestamp) BETWEEN '{START_DATE}' AND '{END_DATE}'
  AND cv.pb.event.name = 'revenue'
  AND cv.pb.moloco.attributed = true
GROUP BY 1, 2
ORDER BY campaign_id, date
""", '2. Campaign-Level Revenue Events (cv)')

if not df_campaign_rev.empty:
    print(f"Unique campaigns with revenue events: {df_campaign_rev['campaign_id'].nunique()}")
    display(df_campaign_rev)

✅ 2. Campaign-Level Revenue Events (cv): 124 rows
Unique campaigns with revenue events: 8


,campaign_id,date,revenue_events,events_with_revenue,pb_revenue
0,BNnzFJ1BQ6sRK9C3,2026-03-01,29,29,158.9461
1,BNnzFJ1BQ6sRK9C3,2026-03-02,19,19,92.7195
2,BNnzFJ1BQ6sRK9C3,2026-03-03,20,20,101.8394
3,BNnzFJ1BQ6sRK9C3,2026-03-04,37,37,147.1363
4,BNnzFJ1BQ6sRK9C3,2026-03-05,18,18,137.0382
...,...,...,...,...,...
119,nqWMPgcAgXjZgEM8,2026-03-12,52,11,74.2704
120,nqWMPgcAgXjZgEM8,2026-03-13,32,5,43.6841
121,nqWMPgcAgXjZgEM8,2026-03-14,40,9,52.1624
122,nqWMPgcAgXjZgEM8,2026-03-15,50,2,10.7311


In [42]:
# Pivot: campaign x date — revenue event count vs actual revenue value
if not df_campaign_rev.empty:
    pivot_event_count = df_campaign_rev.pivot_table(
        index='campaign_id', columns='date', values='revenue_events', aggfunc='sum', fill_value=0
    )
    print("=== Revenue Event Count by Campaign x Date ===")
    display(pivot_event_count)

    pivot_with_rev = df_campaign_rev.pivot_table(
        index='campaign_id', columns='date', values='events_with_revenue', aggfunc='sum', fill_value=0
    )
    print("\n=== Revenue Events WITH Actual Revenue Value by Campaign x Date ===")
    display(pivot_with_rev)

    pivot_rev = df_campaign_rev.pivot_table(
        index='campaign_id', columns='date', values='pb_revenue', aggfunc='sum', fill_value=0
    )
    print("\n=== PB Revenue ($) by Campaign x Date ===")
    display(pivot_rev)

    # Per campaign x date flag
    print("\n=== Campaign Flag by Date: Revenue Events vs Events with Actual Revenue ===")
    flag_df = df_campaign_rev.copy()
    flag_df['pct_with_revenue'] = (flag_df['events_with_revenue'] / flag_df['revenue_events'] * 100).round(1)
    flag_df['flag'] = flag_df.apply(
        lambda r: 'ALL MISSING' if r['events_with_revenue'] == 0
        else 'PARTIAL' if r['events_with_revenue'] < r['revenue_events']
        else 'OK', axis=1
    )
    display(flag_df[['campaign_id', 'date', 'revenue_events', 'events_with_revenue', 'pct_with_revenue', 'pb_revenue', 'flag']].sort_values(['campaign_id', 'date']))

    # Flag pivot: campaign x date
    print("\n=== Flag Pivot: Campaign x Date ===")
    flag_pivot = flag_df.pivot_table(index='campaign_id', columns='date', values='flag', aggfunc='first', fill_value='—')
    display(flag_pivot)

=== Revenue Event Count by Campaign x Date ===


date,2026-03-01,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08,2026-03-09,2026-03-10,2026-03-11,2026-03-12,2026-03-13,2026-03-14,2026-03-15,2026-03-16
campaign_id,,,,,,,,,,,,,,,,
BNnzFJ1BQ6sRK9C3,29,19,20,37,18,45,23,23,32,19,50,207,112,78,83,87
BnyGc57kJEoPrxdW,65,40,48,50,44,44,44,44,55,41,61,127,106,111,55,62
Jl5NUKiajhIyKXQK,0,2,4,0,5,1,7,9,4,5,11,41,22,26,13,22
PcXr8D0MsfIKFEzC,53,10,25,15,9,11,8,25,13,11,10,48,31,21,26,19
VBnWxKikoKxWVkVa,93,92,48,61,26,47,39,59,56,31,132,591,296,295,254,112
ZQUR6yZ4jwpXrTgZ,28,25,27,39,21,27,27,26,25,18,46,133,64,70,55,22
fISFGcvrsN7uXPoR,0,1,0,2,5,1,6,10,10,16,24,60,46,59,49,41
nqWMPgcAgXjZgEM8,7,1,2,6,5,15,19,16,24,12,25,52,32,40,50,40



=== Revenue Events WITH Actual Revenue Value by Campaign x Date ===


date,2026-03-01,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08,2026-03-09,2026-03-10,2026-03-11,2026-03-12,2026-03-13,2026-03-14,2026-03-15,2026-03-16
campaign_id,,,,,,,,,,,,,,,,
BNnzFJ1BQ6sRK9C3,29,19,20,37,18,45,23,23,32,19,50,207,112,78,83,87
BnyGc57kJEoPrxdW,65,40,48,50,44,44,44,44,55,41,61,127,106,111,55,62
Jl5NUKiajhIyKXQK,0,2,4,0,0,0,4,2,1,2,3,1,3,0,0,20
PcXr8D0MsfIKFEzC,53,10,25,15,9,11,8,25,13,11,10,48,31,21,26,19
VBnWxKikoKxWVkVa,93,92,48,61,26,47,39,59,56,31,132,591,296,295,254,112
ZQUR6yZ4jwpXrTgZ,28,25,27,39,21,27,27,26,25,18,46,133,64,70,55,22
fISFGcvrsN7uXPoR,0,1,0,2,3,1,1,5,8,8,13,13,22,12,4,39
nqWMPgcAgXjZgEM8,7,1,2,0,2,9,6,6,1,8,4,11,5,9,2,31



=== PB Revenue ($) by Campaign x Date ===


date,2026-03-01,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08,2026-03-09,2026-03-10,2026-03-11,2026-03-12,2026-03-13,2026-03-14,2026-03-15,2026-03-16
campaign_id,,,,,,,,,,,,,,,,
BNnzFJ1BQ6sRK9C3,158.9461,92.7195,101.8394,147.1363,137.0382,395.3675,88.3374,206.8528,164.2120,91.9374,327.1798,1564.7054,615.4704,433.3208,606.8805,355.1940
BnyGc57kJEoPrxdW,502.2712,221.1941,224.1986,290.6267,219.2946,204.5085,244.7410,198.1414,338.7491,170.2112,376.6535,1092.8700,820.6793,676.6891,338.4212,346.8127
Jl5NUKiajhIyKXQK,0.0000,44.4407,48.4416,0.0000,0.0000,0.0000,14.1458,8.8332,3.8864,6.4713,12.6664,1.6899,5.1081,0.0000,0.0000,78.3132
PcXr8D0MsfIKFEzC,311.8925,90.4167,164.0033,108.7440,41.1114,69.8676,51.8681,148.9303,111.8428,59.4184,75.5767,435.7908,219.7982,192.9602,135.3582,137.7015
VBnWxKikoKxWVkVa,627.1218,657.7230,160.8458,436.8681,120.6015,149.7499,313.9008,341.3544,212.0267,139.7895,745.7686,5703.1350,2611.2427,2637.9588,1829.9466,763.6295
ZQUR6yZ4jwpXrTgZ,167.5800,133.0395,155.0615,200.1615,120.9687,146.1186,139.1793,116.4887,99.6341,78.5633,253.5803,1045.3149,459.2032,554.0347,365.9642,129.3896
fISFGcvrsN7uXPoR,0.0000,1.7776,0.0000,6.2669,2.3764,3.9075,5.8941,22.1418,15.8993,32.7488,60.8668,211.8058,128.0618,83.4565,20.0297,152.2289
nqWMPgcAgXjZgEM8,68.6013,0.6060,4.5520,0.0000,11.8819,42.9965,22.5329,66.7394,2.9443,115.2678,52.5761,74.2704,43.6841,52.1624,10.7311,219.1573



=== Campaign Flag by Date: Revenue Events vs Events with Actual Revenue ===


,campaign_id,date,revenue_events,events_with_revenue,pct_with_revenue,pb_revenue,flag
0,BNnzFJ1BQ6sRK9C3,2026-03-01,29,29,100.0,158.9461,OK
1,BNnzFJ1BQ6sRK9C3,2026-03-02,19,19,100.0,92.7195,OK
2,BNnzFJ1BQ6sRK9C3,2026-03-03,20,20,100.0,101.8394,OK
3,BNnzFJ1BQ6sRK9C3,2026-03-04,37,37,100.0,147.1363,OK
4,BNnzFJ1BQ6sRK9C3,2026-03-05,18,18,100.0,137.0382,OK
...,...,...,...,...,...,...,...
119,nqWMPgcAgXjZgEM8,2026-03-12,52,11,21.2,74.2704,PARTIAL
120,nqWMPgcAgXjZgEM8,2026-03-13,32,5,15.6,43.6841,PARTIAL
121,nqWMPgcAgXjZgEM8,2026-03-14,40,9,22.5,52.1624,PARTIAL
122,nqWMPgcAgXjZgEM8,2026-03-15,50,2,4.0,10.7311,PARTIAL



=== Flag Pivot: Campaign x Date ===


date,2026-03-01,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08,2026-03-09,2026-03-10,2026-03-11,2026-03-12,2026-03-13,2026-03-14,2026-03-15,2026-03-16
campaign_id,,,,,,,,,,,,,,,,
BNnzFJ1BQ6sRK9C3,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
BnyGc57kJEoPrxdW,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
Jl5NUKiajhIyKXQK,—,OK,OK,—,ALL MISSING,ALL MISSING,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,ALL MISSING,ALL MISSING,PARTIAL
PcXr8D0MsfIKFEzC,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
VBnWxKikoKxWVkVa,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
ZQUR6yZ4jwpXrTgZ,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK,OK
fISFGcvrsN7uXPoR,—,OK,—,OK,PARTIAL,OK,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL
nqWMPgcAgXjZgEM8,OK,OK,OK,ALL MISSING,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL


## 3. Campaign Configuration — Goal, Tracking Links, KPI Setup

Check campaign details and tracking link configuration for the 3 suspicious campaigns.

In [43]:
df_campaign_details = run_query(f"""
WITH core AS (
  SELECT
    campaign_id,
    ANY_VALUE(campaign.title) AS title,
    ANY_VALUE(campaign.goal) AS goal,
    ANY_VALUE(campaign.os) AS os,
    ANY_VALUE(campaign.country) AS country,
    MIN(date_utc) AS first_date,
    MAX(date_utc) AS last_date,
    ROUND(SUM(gross_spend_usd), 2) AS total_spend,
    SUM(impressions) AS total_impressions,
    SUM(installs) AS total_installs
  FROM `moloco-ae-view.athena.fact_dsp_core`
  WHERE campaign_id IN ('Jl5NUKiajhIyKXQK', 'fISFGcvrsN7uXPoR', 'nqWMPgcAgXjZgEM8')
    AND date_utc BETWEEN '{START_DATE}' AND '{END_DATE}'
  GROUP BY 1
),
rev AS (
  SELECT
    cv.pb.moloco.campaign_id AS campaign_id,
    COUNT(*) AS total_revenue_events,
    COUNTIF(cv.revenue_usd.amount > 0) AS events_with_revenue,
    ROUND(SUM(IFNULL(cv.revenue_usd.amount, 0)), 2) AS total_revenue_usd
  FROM `focal-elf-631.prod_stream_view.cv`
  WHERE cv.pb.moloco.campaign_id IN ('Jl5NUKiajhIyKXQK', 'fISFGcvrsN7uXPoR', 'nqWMPgcAgXjZgEM8')
    AND DATE(timestamp) BETWEEN '{START_DATE}' AND '{END_DATE}'
    AND cv.pb.event.name = 'revenue'
    AND cv.pb.moloco.attributed = true
  GROUP BY 1
)
SELECT c.*, r.total_revenue_events, r.events_with_revenue, r.total_revenue_usd
FROM core c
LEFT JOIN rev r USING (campaign_id)
""", '3. Campaign Details')

if not df_campaign_details.empty:
    display(df_campaign_details)

✅ 3. Campaign Details: 3 rows


,campaign_id,title,goal,os,country,first_date,last_date,total_spend,total_impressions,total_installs,total_revenue_events,events_with_revenue,total_revenue_usd
0,fISFGcvrsN7uXPoR,WISEBIRDS_[XRTX]sololv_update_moloco_KR_And_tR...,OPTIMIZE_ROAS_FOR_APP_RE,ANDROID,KOR,2026-03-01,2026-03-16,36111.320000000,7025631,1,330,132,747.46
1,nqWMPgcAgXjZgEM8,WISEBIRDS_[XRTX]sololv_update_moloco_US_And_AC...,OPTIMIZE_CPA_FOR_APP_RE,ANDROID,USA,2026-03-01,2026-03-16,38589.880000000,6012253,3,346,104,788.70
2,Jl5NUKiajhIyKXQK,WISEBIRDS_[XRTX]sololv_update_moloco_WW_And_tR...,OPTIMIZE_ROAS_FOR_APP_RE,ANDROID,HUN,2026-03-01,2026-03-16,27302.760000000,25333512,128,172,42,224.00


In [44]:
df_event_config = run_query(f"""
SELECT
  cv.pb.moloco.campaign_id AS campaign_id,
  cv.pb.event.name AS event_name,
  COUNT(*) AS event_count,
  COUNTIF(cv.revenue_usd.amount > 0) AS with_revenue,
  ROUND(SUM(IFNULL(cv.revenue_usd.amount, 0)), 4) AS total_revenue
FROM `focal-elf-631.prod_stream_view.cv`
WHERE cv.pb.moloco.campaign_id IN ('Jl5NUKiajhIyKXQK', 'fISFGcvrsN7uXPoR', 'nqWMPgcAgXjZgEM8')
  AND DATE(timestamp) BETWEEN '{START_DATE}' AND '{END_DATE}'
  AND cv.pb.moloco.attributed = true
GROUP BY 1, 2
HAVING event_count > 0
ORDER BY campaign_id, event_count DESC
""", '3b. Event Config per Campaign (cv)')

if not df_event_config.empty:
    display(df_event_config)

✅ 3b. Event Config per Campaign (cv): 135 rows


,campaign_id,event_name,event_count,with_revenue,total_revenue
0,Jl5NUKiajhIyKXQK,af_app_opened,421663,0,0.0
1,Jl5NUKiajhIyKXQK,login,273460,0,0.0
2,Jl5NUKiajhIyKXQK,visit_shop,225159,0,0.0
3,Jl5NUKiajhIyKXQK,gacha_normal_x10,144080,0,0.0
4,Jl5NUKiajhIyKXQK,cdn_download_finished,131642,0,0.0
...,...,...,...,...,...
130,nqWMPgcAgXjZgEM8,clear_chpt4,124,0,0.0
131,nqWMPgcAgXjZgEM8,tutorial_complete,123,0,0.0
132,nqWMPgcAgXjZgEM8,lvachieved_user_10,70,0,0.0
133,nqWMPgcAgXjZgEM8,lvachieved_user_15,63,0,0.0


## 3b. Raw Postback: Attributed vs Unattributed Revenue (KEY SECTION)

Compare revenue events in `df_accesslog.pb` **before** Moloco attribution processing.  
If unattributed postbacks also have $0 revenue → MMP-side problem.  
If only attributed postbacks show $0 → attribution/processing issue.

In [45]:
df_pb_revenue = run_query(f"""
SELECT
  DATE(timestamp) AS date,
  attribution.attributed,
  CASE
    WHEN event.revenue_usd.amount > 0 THEN 'has_revenue'
    WHEN event.revenue_usd.amount = 0 THEN 'zero_revenue'
    ELSE 'null_revenue'
  END AS revenue_status,
  COUNT(*) AS event_count,
  ROUND(SUM(IFNULL(event.revenue_usd.amount, 0)), 4) AS total_revenue_usd
FROM `focal-elf-631.df_accesslog.pb`
WHERE DATE(timestamp) BETWEEN '{START_DATE}' AND '{END_DATE}'
  AND app.bundle = 'com.netmarble.sololv'
  AND event.name = 'revenue'
GROUP BY 1, 2, 3
ORDER BY date, attributed DESC, revenue_status
""", '3b. Raw PB Attributed vs Unattributed')

if not df_pb_revenue.empty:
    print("=== Aggregated (All Dates) ===")
    df_pb_agg = df_pb_revenue.groupby(['attributed', 'revenue_status'], as_index=False).agg(
        event_count=('event_count', 'sum'),
        total_revenue_usd=('total_revenue_usd', 'sum')
    ).sort_values(['attributed', 'revenue_status'], ascending=[False, True])
    display(df_pb_agg)

    print("\n=== By Date ===")
    display(df_pb_revenue)

    print("\n=== Pivot: Daily Revenue by Attribution Status ===")
    pivot = df_pb_revenue.groupby(['date', 'attributed']).agg(
        events=('event_count', 'sum'),
        revenue=('total_revenue_usd', 'sum')
    ).unstack(fill_value=0)
    display(pivot)

    print("\n=== Attributed Only: Revenue Status by Date ===")
    df_attr = df_pb_revenue[df_pb_revenue['attributed'] == True]
    if not df_attr.empty:
        display(df_attr[['date', 'revenue_status', 'event_count', 'total_revenue_usd']])
    else:
        print("No attributed revenue events found.")

    print("\n=== Unattributed Only: Revenue Status by Date ===")
    df_unattr = df_pb_revenue[df_pb_revenue['attributed'] == False]
    if not df_unattr.empty:
        display(df_unattr[['date', 'revenue_status', 'event_count', 'total_revenue_usd']])
    else:
        print("No unattributed revenue events found.")

✅ 3b. Raw PB Attributed vs Unattributed: 45 rows
=== Aggregated (All Dates) ===


,attributed,revenue_status,event_count,total_revenue_usd
1,True,has_revenue,10512,72712.4730
2,True,zero_revenue,570,0.0000
0,False,has_revenue,90544,587688.5029



=== By Date ===


,date,attributed,revenue_status,event_count,total_revenue_usd
0,2026-03-01,True,has_revenue,495,3272.9441
1,2026-03-01,False,has_revenue,4188,26269.0033
2,2026-03-02,True,has_revenue,365,2332.5328
3,2026-03-02,False,has_revenue,3637,18960.1168
4,2026-03-03,True,has_revenue,330,1868.6812
5,2026-03-03,False,has_revenue,2760,14713.2465
6,2026-03-04,True,has_revenue,387,2496.3369
7,2026-03-04,True,zero_revenue,6,0.0000
8,2026-03-04,False,has_revenue,2706,14480.0511
9,2026-03-05,True,has_revenue,271,1584.1682



=== Pivot: Daily Revenue by Attribution Status ===


events            revenue            
attributed  False True         False       True 
date                                            
2026-03-01   4188   495   26269.0033   3272.9441
2026-03-02   3637   365   18960.1168   2332.5328
2026-03-03   2760   330   14713.2465   1868.6812
2026-03-04   2706   393   14480.0511   2496.3369
2026-03-05   2625   281   14880.8253   1584.1682
2026-03-06   2647   325   14915.0478   1925.8928
2026-03-07   2631   330   14857.9604   2017.7435
2026-03-08   3069   411   16852.4495   2309.2538
2026-03-09   3303   372   16169.2978   1890.8542
2026-03-10   2439   288   12153.7580   1476.3080
2026-03-11   5080   658   29259.3867   3578.7791
2026-03-12  19189  2268  158610.9308  18404.4170
2026-03-13  11279  1387   81518.3231  10020.3845
2026-03-14  10120  1355   67052.6469   9155.8009
2026-03-15   7887  1031   48164.9475   6152.5521
2026-03-16   6984   793   38830.5114   4225.8239


=== Attributed Only: Revenue Status by Date ===


,date,revenue_status,event_count,total_revenue_usd
0,2026-03-01,has_revenue,495,3272.9441
2,2026-03-02,has_revenue,365,2332.5328
4,2026-03-03,has_revenue,330,1868.6812
6,2026-03-04,has_revenue,387,2496.3369
7,2026-03-04,zero_revenue,6,0.0000
9,2026-03-05,has_revenue,271,1584.1682
10,2026-03-05,zero_revenue,10,0.0000
12,2026-03-06,has_revenue,318,1925.8928
13,2026-03-06,zero_revenue,7,0.0000
15,2026-03-07,has_revenue,309,2017.7435



=== Unattributed Only: Revenue Status by Date ===


,date,revenue_status,event_count,total_revenue_usd
1,2026-03-01,has_revenue,4188,26269.0033
3,2026-03-02,has_revenue,3637,18960.1168
5,2026-03-03,has_revenue,2760,14713.2465
8,2026-03-04,has_revenue,2706,14480.0511
11,2026-03-05,has_revenue,2625,14880.8253
14,2026-03-06,has_revenue,2647,14915.0478
17,2026-03-07,has_revenue,2631,14857.9604
20,2026-03-08,has_revenue,3069,16852.4495
23,2026-03-09,has_revenue,3303,16169.2978
26,2026-03-10,has_revenue,2439,12153.7580


In [47]:
df_pb_by_campaign = run_query(f"""
SELECT
  DATE(timestamp) AS date,
  moloco.campaign_id,
  CASE
    WHEN event.revenue_usd.amount > 0 THEN 'has_revenue'
    WHEN event.revenue_usd.amount = 0 THEN 'zero_revenue'
    ELSE 'null_revenue'
  END AS revenue_status,
  COUNT(*) AS event_count,
  ROUND(SUM(IFNULL(event.revenue_usd.amount, 0)), 4) AS total_revenue_usd
FROM `focal-elf-631.df_accesslog.pb`
WHERE DATE(timestamp) BETWEEN '{START_DATE}' AND '{END_DATE}'
  AND app.bundle = 'com.netmarble.sololv'
  AND event.name = 'revenue'
  AND attribution.attributed = true
  AND moloco.campaign_id IN ('Jl5NUKiajhIyKXQK', 'fISFGcvrsN7uXPoR', 'nqWMPgcAgXjZgEM8')
GROUP BY 1, 2, 3
ORDER BY campaign_id, date, revenue_status
""", '3b. Campaign-Level Raw PB')

if not df_pb_by_campaign.empty:
    print("=== Aggregated (All Dates) ===")
    df_camp_agg = df_pb_by_campaign.groupby(['campaign_id', 'revenue_status'], as_index=False).agg(
        event_count=('event_count', 'sum'),
        total_revenue_usd=('total_revenue_usd', 'sum')
    ).sort_values(['campaign_id', 'revenue_status'])
    display(df_camp_agg)

    print("\n=== By Date ===")
    display(df_pb_by_campaign)

    print("\n=== Pivot: Revenue per Campaign x Date ===")
    pivot_camp = df_pb_by_campaign.groupby(['campaign_id', 'date']).agg(
        events=('event_count', 'sum'),
        revenue=('total_revenue_usd', 'sum')
    ).unstack(fill_value=0)
    display(pivot_camp)

✅ 3b. Campaign-Level Raw PB: 75 rows
=== Aggregated (All Dates) ===


,campaign_id,revenue_status,event_count,total_revenue_usd
0,Jl5NUKiajhIyKXQK,has_revenue,42,223.9966
1,Jl5NUKiajhIyKXQK,zero_revenue,130,0.0000
2,fISFGcvrsN7uXPoR,has_revenue,132,747.4619
3,fISFGcvrsN7uXPoR,zero_revenue,198,0.0000
4,nqWMPgcAgXjZgEM8,has_revenue,104,788.7035
5,nqWMPgcAgXjZgEM8,zero_revenue,242,0.0000



=== By Date ===


,date,campaign_id,revenue_status,event_count,total_revenue_usd
0,2026-03-02,Jl5NUKiajhIyKXQK,has_revenue,2,44.4407
1,2026-03-03,Jl5NUKiajhIyKXQK,has_revenue,4,48.4416
2,2026-03-05,Jl5NUKiajhIyKXQK,zero_revenue,5,0.0000
3,2026-03-06,Jl5NUKiajhIyKXQK,zero_revenue,1,0.0000
4,2026-03-07,Jl5NUKiajhIyKXQK,has_revenue,4,14.1458
5,2026-03-07,Jl5NUKiajhIyKXQK,zero_revenue,3,0.0000
6,2026-03-08,Jl5NUKiajhIyKXQK,has_revenue,2,8.8332
7,2026-03-08,Jl5NUKiajhIyKXQK,zero_revenue,7,0.0000
8,2026-03-09,Jl5NUKiajhIyKXQK,has_revenue,1,3.8864
9,2026-03-09,Jl5NUKiajhIyKXQK,zero_revenue,3,0.0000



=== Pivot: Revenue per Campaign x Date ===


events                                              \
date             2026-03-01 2026-03-02 2026-03-03 2026-03-04 2026-03-05   
campaign_id                                                               
Jl5NUKiajhIyKXQK          0          2          4          0          5   
fISFGcvrsN7uXPoR          0          1          0          2          5   
nqWMPgcAgXjZgEM8          7          1          2          6          5   

                                                                         \
date             2026-03-06 2026-03-07 2026-03-08 2026-03-09 2026-03-10   
campaign_id                                                               
Jl5NUKiajhIyKXQK          1          7          9          4          5   
fISFGcvrsN7uXPoR          1          6         10         10         16   
nqWMPgcAgXjZgEM8         15         19         16         24         12   

                                                                         \
date             2026-03-11 2026-03-12 2026-03-13 2026-03-14 2026-03-15   
campaign_id                                                               
Jl5NUKiajhIyKXQK         11         41         22         26         13   
fISFGcvrsN7uXPoR         24         60         46         59         49   
nqWMPgcAgXjZgEM8         25         52         32         40         50   

                               revenue                                   \
date             2026-03-16 2026-03-01 2026-03-02 2026-03-03 2026-03-04   
campaign_id                                                               
Jl5NUKiajhIyKXQK         22     0.0000    44.4407    48.4416     0.0000   
fISFGcvrsN7uXPoR         41     0.0000     1.7776     0.0000     6.2669   
nqWMPgcAgXjZgEM8         40    68.6013     0.6060     4.5520     0.0000   

                                                                         \
date             2026-03-05 2026-03-06 2026-03-07 2026-03-08 2026-03-09   
campaign_id                                                               
Jl5NUKiajhIyKXQK     0.0000     0.0000    14.1458     8.8332     3.8864   
fISFGcvrsN7uXPoR     2.3764     3.9075     5.8941    22.1418    15.8993   
nqWMPgcAgXjZgEM8    11.8819    42.9965    22.5329    66.7394     2.9443   

                                                                         \
date             2026-03-10 2026-03-11 2026-03-12 2026-03-13 2026-03-14   
campaign_id                                                               
Jl5NUKiajhIyKXQK     6.4713    12.6664     1.6899     5.1081     0.0000   
fISFGcvrsN7uXPoR    32.7488    60.8668   211.8058   128.0618    83.4565   
nqWMPgcAgXjZgEM8   115.2678    52.5761    74.2704    43.6841    52.1624   

                                        
date             2026-03-15 2026-03-16  
campaign_id                             
Jl5NUKiajhIyKXQK     0.0000    78.3132  
fISFGcvrsN7uXPoR    20.0297   152.2289  
nqWMPgcAgXjZgEM8    10.7311   219.1573

## 3b-2. Wisebirds Campaigns Only — Revenue Status by Campaign (`af_prt=wisebirds`)

Same campaign-level breakdown as 3b, scoped to postbacks from the 3 active Wisebirds-managed campaigns.  
**Note**: `af_prt` is not a parsed field in `df_accesslog.pb` — it exists only in `pb_raw.request.url`.  
Since the 3 campaign IDs specified are exclusively Wisebirds campaigns (confirmed via raw URL: `af_prt=wisebirds`),  
filtering by campaign ID is semantically equivalent to filtering `af_prt=wisebirds`.

**Key metric**: `zero_pct` — share of revenue postbacks carrying no revenue amount (MMP sent `N/A`).

In [4]:
# NOTE: pb_raw has a 5-day TTL — use a recent date range only.
# We explicitly filter request.url for af_prt=wisebirds to confirm MMP partner tag.
PB_RAW_START = '2026-03-01'
PB_RAW_END   = '2026-03-15'

df_wb_campaign = run_query(f"""
SELECT
  DATE(DATETIME(timestamp, 'Asia/Seoul')) AS date,
  moloco.campaign_id,
  CASE
    WHEN event.revenue_usd.amount > 0 THEN 'has_revenue'
    WHEN event.revenue_usd.amount = 0 THEN 'zero_revenue'
    ELSE 'null_revenue'
  END AS revenue_status,
  COUNT(*) AS event_count,
  ROUND(SUM(IFNULL(event.revenue_usd.amount, 0)), 4) AS total_revenue_usd
FROM `focal-elf-631.df_accesslog.pb_raw`
WHERE DATE(DATETIME(timestamp, 'Asia/Seoul')) BETWEEN '{PB_RAW_START}' AND '{PB_RAW_END}'
  AND app.bundle = 'com.netmarble.sololv'
  AND event.name = 'revenue'
  AND attribution.attributed = true
  AND moloco.campaign_id IN ('Jl5NUKiajhIyKXQK', 'fISFGcvrsN7uXPoR', 'nqWMPgcAgXjZgEM8')
  AND REGEXP_CONTAINS(request.url, r'(?:^|[?&])af_prt=wisebirds(?:&|$)')
GROUP BY 1, 2, 3
ORDER BY campaign_id, date, revenue_status
""", '3b-2. Wisebirds Campaign Revenue Status (pb_raw, af_prt=wisebirds)')

if not df_wb_campaign.empty:
    print("=== Aggregated (All Dates) + Zero % ===")
    df_wb_agg = df_wb_campaign.groupby(['campaign_id', 'revenue_status'], as_index=False).agg(
        event_count=('event_count', 'sum'),
        total_revenue_usd=('total_revenue_usd', 'sum')
    )
    total_per_camp = df_wb_agg.groupby('campaign_id')['event_count'].transform('sum')
    df_wb_agg['zero_pct'] = (
        df_wb_agg['event_count'] / total_per_camp * 100
    ).where(df_wb_agg['revenue_status'] != 'has_revenue', 0).round(1)
    display(df_wb_agg.sort_values(['campaign_id', 'revenue_status']))

    print("\n=== Zero Revenue Rate Summary per Campaign ===")
    df_zero_rate = df_wb_campaign.groupby('campaign_id').apply(
        lambda g: pd.Series({
            'total_events':     g['event_count'].sum(),
            'has_revenue':      g.loc[g['revenue_status'] == 'has_revenue', 'event_count'].sum(),
            'zero_or_null':     g.loc[g['revenue_status'] != 'has_revenue', 'event_count'].sum(),
            'total_revenue_usd': g['total_revenue_usd'].sum()
        })
    ).reset_index()
    df_zero_rate['zero_pct'] = (df_zero_rate['zero_or_null'] / df_zero_rate['total_events'] * 100).round(1)
    display(df_zero_rate)

    print("\n=== Daily By Campaign ===")
    display(df_wb_campaign)

    print("\n=== Pivot: Daily Revenue per Campaign ===")
    pivot_wb = df_wb_campaign.groupby(['campaign_id', 'date']).agg(
        events=('event_count', 'sum'),
        revenue=('total_revenue_usd', 'sum')
    ).unstack(fill_value=0)
    display(pivot_wb)

✅ 3b-2. Wisebirds Campaign Revenue Status (pb_raw, af_prt=wisebirds): 6 rows
=== Aggregated (All Dates) + Zero % ===


,campaign_id,revenue_status,event_count,total_revenue_usd,zero_pct
0,Jl5NUKiajhIyKXQK,zero_revenue,33,0.0,100.0
1,fISFGcvrsN7uXPoR,zero_revenue,77,0.0,100.0
2,nqWMPgcAgXjZgEM8,zero_revenue,61,0.0,100.0



=== Zero Revenue Rate Summary per Campaign ===


,campaign_id,total_events,has_revenue,zero_or_null,total_revenue_usd,zero_pct
0,Jl5NUKiajhIyKXQK,33.0,0.0,33.0,0.0,100.0
1,fISFGcvrsN7uXPoR,77.0,0.0,77.0,0.0,100.0
2,nqWMPgcAgXjZgEM8,61.0,0.0,61.0,0.0,100.0



=== Daily By Campaign ===


,date,campaign_id,revenue_status,event_count,total_revenue_usd
0,2026-03-14,Jl5NUKiajhIyKXQK,zero_revenue,17,0.0
1,2026-03-15,Jl5NUKiajhIyKXQK,zero_revenue,16,0.0
2,2026-03-14,fISFGcvrsN7uXPoR,zero_revenue,30,0.0
3,2026-03-15,fISFGcvrsN7uXPoR,zero_revenue,47,0.0
4,2026-03-14,nqWMPgcAgXjZgEM8,zero_revenue,22,0.0
5,2026-03-15,nqWMPgcAgXjZgEM8,zero_revenue,39,0.0



=== Pivot: Daily Revenue per Campaign ===


events               revenue           
date             2026-03-14 2026-03-15 2026-03-14 2026-03-15
campaign_id                                                 
Jl5NUKiajhIyKXQK         17         16        0.0        0.0
fISFGcvrsN7uXPoR         30         47        0.0        0.0
nqWMPgcAgXjZgEM8         22         39        0.0        0.0

## 3c. Comparison: Raw Postback vs Attributed Table (fact_dsp_all)

Cross-reference raw postback revenue with the attributed table to determine where the gap occurs.

In [ ]:
df_attr_daily = run_query(f"""
SELECT
  DATE(timestamp) AS date,
  COUNT(*) AS attr_revenue_events,
  COUNTIF(cv.revenue_usd.amount > 0) AS attr_with_revenue,
  ROUND(SUM(IFNULL(cv.revenue_usd.amount, 0)), 4) AS attr_revenue_usd
FROM `focal-elf-631.prod_stream_view.cv`
WHERE cv.pb.app.bundle = 'com.netmarble.sololv'
  AND DATE(timestamp) BETWEEN '{START_DATE}' AND '{END_DATE}'
  AND cv.pb.event.name = 'revenue'
  AND cv.pb.moloco.attributed = true
GROUP BY 1
ORDER BY 1
""", '3c. Attributed Daily Revenue (cv)')

if not df_attr_daily.empty and not df_pb_revenue.empty:
    pb_daily = df_pb_revenue.groupby('date').agg(
        raw_events=('event_count', 'sum'),
        raw_revenue=('total_revenue_usd', 'sum')
    ).reset_index()
    pb_daily['date'] = pb_daily['date'].astype(str)
    df_attr_daily['date'] = df_attr_daily['date'].astype(str)

    comparison = pd.merge(df_attr_daily, pb_daily, on='date', how='outer').sort_values('date')
    comparison['gap'] = comparison['raw_revenue'].fillna(0) - comparison['attr_revenue_usd'].fillna(0)
    print("=== Comparison: Attributed (cv) vs Raw Postback (df_accesslog.pb) ===")
    print("If raw_revenue ≈ attr_revenue_usd → revenue passes through correctly")
    print("If raw_revenue > 0 but attr = 0 → revenue lost during processing")
    print("If both = 0 → MMP never sent revenue\n")
    display(comparison)
elif not df_attr_daily.empty:
    print("=== Attributed Daily Revenue (raw postback query not available) ===")
    display(df_attr_daily)

✅ 3c. Attributed Daily Revenue (cv): 18 rows
=== Comparison: Attributed (cv) vs Raw Postback (df_accesslog.pb) ===
If raw_revenue ≈ attr_revenue_usd → revenue passes through correctly
If raw_revenue > 0 but attr = 0 → revenue lost during processing
If both = 0 → MMP never sent revenue



,date,attr_revenue_events,attr_with_revenue,attr_revenue_usd,raw_events,raw_revenue,gap
0,2026-03-01,275,275,1836.4130,4683,29541.9474,27705.5344
1,2026-03-02,190,190,1241.9172,4002,21292.6496,20050.7324
2,2026-03-03,174,174,858.9420,3090,16581.9277,15722.9857
3,2026-03-04,210,204,1189.8034,3099,16976.3880,15786.5846
4,2026-03-05,133,123,653.2728,2906,16464.9935,15811.7207
5,2026-03-06,191,184,1012.5161,2972,16840.9406,15828.4245
6,2026-03-07,173,152,880.5994,2961,16875.7039,15995.1045
7,2026-03-08,212,190,1109.4819,3480,19161.7033,18052.2214
8,2026-03-09,219,191,949.1946,3675,18060.1520,17110.9574
9,2026-03-10,153,138,694.4077,2727,13630.0660,12935.6583


## 4. Advertiser-Level Analysis — `yfg0At8VksGnt6EO` & `tGYMnmOyEZqjEIND`

Replicates Sections 2–3c for **all campaigns** under two advertiser IDs.  
Date range: 2026-03-01 to 2026-03-10.

- **Step 4a**: Campaign discovery
- **Step 4b**: Campaign-level attributed revenue events
- **Step 4c**: Raw postback (attributed vs unattributed)
- **Step 4d**: Comparison: cv vs raw postback

In [ ]:
ADVERTISER_IDS = ['yfg0At8VksGnt6EO', 'tGYMnmOyEZqjEIND']
ADV_FILTER = ', '.join(f"'{a}'" for a in ADVERTISER_IDS)

df_campaigns = run_query(f"""
SELECT
  advertiser_id,
  campaign_id,
  ANY_VALUE(campaign.title) AS title,
  ANY_VALUE(campaign.goal) AS goal,
  ANY_VALUE(campaign.os) AS os,
  ANY_VALUE(campaign.country) AS country,
  MIN(date_utc) AS first_date,
  MAX(date_utc) AS last_date,
  ROUND(SUM(gross_spend_usd), 2) AS total_spend,
  SUM(impressions) AS total_impressions,
  SUM(installs) AS total_installs
FROM `moloco-ae-view.athena.fact_dsp_core`
WHERE advertiser_id IN ({ADV_FILTER})
  AND date_utc BETWEEN '2026-03-01' AND '2026-03-10'
GROUP BY 1, 2
HAVING total_spend > 0
ORDER BY advertiser_id, total_spend DESC
""", '4a. Campaign Discovery (spend > 0)')

if not df_campaigns.empty:
    for adv_id in ADVERTISER_IDS:
        n = len(df_campaigns[df_campaigns.advertiser_id == adv_id])
        print(f'Advertiser {adv_id}: {n} campaigns with spend')
    display(df_campaigns)

CAMPAIGN_IDS = df_campaigns['campaign_id'].tolist() if not df_campaigns.empty else []
print(f'\nTotal campaigns to analyze: {len(CAMPAIGN_IDS)}')

✅ 4a. Campaign Discovery (spend > 0): 35 rows
Advertiser yfg0At8VksGnt6EO: 32 campaigns with spend
Advertiser tGYMnmOyEZqjEIND: 3 campaigns with spend


,advertiser_id,campaign_id,title,goal,os,country,first_date,last_date,total_spend,total_impressions,total_installs
0,tGYMnmOyEZqjEIND,fISFGcvrsN7uXPoR,WISEBIRDS_[XRTX]sololv_update_moloco_KR_And_tR...,OPTIMIZE_ROAS_FOR_APP_RE,ANDROID,KOR,2026-03-01,2026-03-10,23546.650000000,4456076,1
1,tGYMnmOyEZqjEIND,nqWMPgcAgXjZgEM8,WISEBIRDS_[XRTX]sololv_update_moloco_US_And_AC...,OPTIMIZE_CPA_FOR_APP_RE,ANDROID,USA,2026-03-01,2026-03-10,16176.840000000,3352742,3
2,tGYMnmOyEZqjEIND,Jl5NUKiajhIyKXQK,WISEBIRDS_[XRTX]sololv_update_moloco_WW_And_tR...,OPTIMIZE_ROAS_FOR_APP_RE,ANDROID,MNG,2026-03-01,2026-03-10,10399.200000000,14882129,76
3,yfg0At8VksGnt6EO,nazpxG3J5MareHRz,[NEW]stonekey_launch_moloco_KR_And_tROAS_260303,OPTIMIZE_ROAS_FOR_APP_UA,ANDROID,KOR,2026-03-03,2026-03-10,156630.160000000,93558251,6608
4,yfg0At8VksGnt6EO,ylgO8XQvDb5nx3k4,[NEW]stonekey_launch_moloco_KR_And_login_1st_2...,OPTIMIZE_CPA_FOR_APP_UA,ANDROID,KOR,2026-03-03,2026-03-10,102248.850000000,86118303,9125
5,yfg0At8VksGnt6EO,EQCWerD5mEThZO4P,[NEW]stonekey_launch_moloco_KR_And_AEO(buy_pet...,OPTIMIZE_CPA_FOR_APP_UA,ANDROID,KOR,2026-03-03,2026-03-10,99692.540000000,47646050,7185
6,yfg0At8VksGnt6EO,GdPe1hm9tPMaUhbt,[CKRC]stonekey_launch_moloco_KR_iOS_login_260303,OPTIMIZE_CPA_FOR_APP_UA,IOS,KOR,2026-03-03,2026-03-10,50493.670000000,63714028,4757
7,yfg0At8VksGnt6EO,huSii8ixFBbQjDXt,[NEW]stonekey_launch_moloco_WW3_And_tROAS_260304,OPTIMIZE_ROAS_FOR_APP_UA,ANDROID,NOR,2026-03-04,2026-03-10,33530.130000000,9436420,3661
8,yfg0At8VksGnt6EO,emorTW8Y48l92YcV,[CTWC]stonekey_launch_moloco_TW_iOS_login_260303,OPTIMIZE_CPA_FOR_APP_UA,IOS,TWN,2026-03-03,2026-03-10,30573.930000000,9905412,1145
9,yfg0At8VksGnt6EO,unbUzFob1WCmTrVD,[NEW]stonekey_launch_moloco_WW1_And_tROAS_260303,OPTIMIZE_ROAS_FOR_APP_UA,ANDROID,HUN,2026-03-03,2026-03-10,23291.820000000,8176110,1301



Total campaigns to analyze: 35


### 4b. Campaign-Level Attributed Revenue Events

Daily revenue event count and revenue amount per campaign.

In [ ]:
if CAMPAIGN_IDS:
    camp_filter = ', '.join(f"'{c}'" for c in CAMPAIGN_IDS)
    camp_to_adv = df_campaigns.set_index('campaign_id')['advertiser_id'].to_dict()

    df_adv_campaign_rev = run_query(f"""
    SELECT
      cv.pb.moloco.campaign_id AS campaign_id,
      DATE(timestamp) AS date,
      COUNT(*) AS revenue_events,
      COUNTIF(cv.revenue_usd.amount > 0) AS events_with_revenue,
      ROUND(SUM(IFNULL(cv.revenue_usd.amount, 0)), 4) AS pb_revenue
    FROM `focal-elf-631.prod_stream_view.cv`
    WHERE cv.pb.moloco.campaign_id IN ({camp_filter})
      AND DATE(timestamp) BETWEEN '2026-03-01' AND '2026-03-10'
      AND cv.pb.event.name = 'revenue'
      AND cv.pb.moloco.attributed = true
    GROUP BY 1, 2
    ORDER BY campaign_id, date
    """, '4b. Campaign-Level Revenue Events (cv)')

    if not df_adv_campaign_rev.empty:
        df_adv_campaign_rev['advertiser_id'] = df_adv_campaign_rev['campaign_id'].map(camp_to_adv)
        df_adv_campaign_rev['pct_with_revenue'] = (
            df_adv_campaign_rev['events_with_revenue'] / df_adv_campaign_rev['revenue_events'] * 100
        ).round(1)
        df_adv_campaign_rev['flag'] = df_adv_campaign_rev.apply(
            lambda r: 'ALL MISSING' if r['events_with_revenue'] == 0
            else 'PARTIAL' if r['events_with_revenue'] < r['revenue_events']
            else 'OK', axis=1
        )

        for adv_id in ADVERTISER_IDS:
            sub = df_adv_campaign_rev[df_adv_campaign_rev['advertiser_id'] == adv_id]
            if sub.empty:
                print(f'\n{adv_id}: no revenue events found.')
                continue
            print(f'\n=== Advertiser: {adv_id} ===')
            print(f'Campaigns with revenue events: {sub["campaign_id"].nunique()}')

            pivot_rev = sub.pivot_table(
                index='campaign_id', columns='date', values='pb_revenue', aggfunc='sum', fill_value=0
            )
            print('Revenue ($) by Campaign x Date:')
            display(pivot_rev)

            flag_pivot = sub.pivot_table(
                index='campaign_id', columns='date', values='flag', aggfunc='first', fill_value='—'
            )
            print('Flag (OK / PARTIAL / ALL MISSING) by Campaign x Date:')
            display(flag_pivot)
    else:
        df_adv_campaign_rev = pd.DataFrame()
else:
    print('No campaigns found — skipping 4b.')
    df_adv_campaign_rev = pd.DataFrame()

✅ 4b. Campaign-Level Revenue Events (cv): 264 rows

=== Advertiser: yfg0At8VksGnt6EO ===
Campaigns with revenue events: 31
Revenue ($) by Campaign x Date:


date,2026-03-01,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08,2026-03-09,2026-03-10
campaign_id,,,,,,,,,,
EQCWerD5mEThZO4P,0.0000,0.0000,2314.7662,1634.0492,3178.6854,2129.8447,2292.0368,1702.3460,1268.5090,1049.0785
EcQ1HKvPWtQZkoEg,0.0000,0.0000,0.0000,0.0000,11.0754,8.8808,57.2121,53.8653,22.1801,0.5883
GdPe1hm9tPMaUhbt,0.0000,0.0000,4196.3126,5784.3438,4774.9978,4069.6396,3989.6287,3930.7526,4197.5480,2841.6062
HvtBaZCEslMI1rJf,0.0000,0.0000,57.4756,123.6888,179.2032,197.6790,216.2073,189.8953,123.8162,70.3244
I1UFqtzeIhRkfCrK,0.0000,0.0000,16.3020,18.4459,32.5169,22.0822,4.1259,12.1335,30.4241,3.8864
KEfora9BuIXZ6T2X,7.2727,1.2120,50.3808,2.9561,97.0357,0.0000,0.0000,6.8310,0.5889,0.0000
MDGEIQDQ44rmAJGF,0.0000,0.0000,3.9858,6.7138,6.5351,35.4442,22.7905,60.4188,1.1777,3.8828
MsUKwNvjuzg3goIw,0.0000,0.0000,2.1922,1.1891,0.0000,3.7497,0.1965,0.0000,0.0000,0.0000
NMrtO3Y5EN9EAiYm,0.0000,0.0000,0.0000,1.1824,8.8322,43.2260,5.1082,10.6025,16.2525,26.6697


Flag (OK / PARTIAL / ALL MISSING) by Campaign x Date:


date,2026-03-01,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08,2026-03-09,2026-03-10
campaign_id,,,,,,,,,,
EQCWerD5mEThZO4P,—,—,OK,OK,OK,OK,OK,OK,OK,OK
EcQ1HKvPWtQZkoEg,—,—,—,—,OK,OK,OK,OK,OK,OK
GdPe1hm9tPMaUhbt,—,—,OK,OK,OK,OK,OK,OK,OK,OK
HvtBaZCEslMI1rJf,—,—,OK,OK,OK,OK,OK,OK,OK,OK
I1UFqtzeIhRkfCrK,—,—,OK,OK,OK,OK,OK,OK,OK,OK
KEfora9BuIXZ6T2X,OK,OK,OK,OK,OK,—,—,OK,OK,—
MDGEIQDQ44rmAJGF,—,—,OK,OK,OK,OK,OK,OK,OK,OK
MsUKwNvjuzg3goIw,—,—,OK,OK,—,OK,OK,—,—,—
NMrtO3Y5EN9EAiYm,—,—,—,OK,OK,OK,OK,OK,OK,OK



=== Advertiser: tGYMnmOyEZqjEIND ===
Campaigns with revenue events: 3
Revenue ($) by Campaign x Date:


date,2026-03-01,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08,2026-03-09,2026-03-10
campaign_id,,,,,,,,,,
Jl5NUKiajhIyKXQK,0.0000,44.4407,48.4416,0.0000,0.0000,0.0000,14.1458,8.8332,3.8864,6.4713
fISFGcvrsN7uXPoR,0.0000,1.7776,0.0000,6.2669,2.3764,3.9075,5.8941,22.1418,15.8993,32.7488
nqWMPgcAgXjZgEM8,68.6013,0.6060,4.5520,0.0000,11.8819,42.9965,22.5329,66.7394,2.9443,115.2678


Flag (OK / PARTIAL / ALL MISSING) by Campaign x Date:


date,2026-03-01,2026-03-02,2026-03-03,2026-03-04,2026-03-05,2026-03-06,2026-03-07,2026-03-08,2026-03-09,2026-03-10
campaign_id,,,,,,,,,,
Jl5NUKiajhIyKXQK,—,OK,OK,—,ALL MISSING,ALL MISSING,PARTIAL,PARTIAL,PARTIAL,PARTIAL
fISFGcvrsN7uXPoR,—,OK,—,OK,PARTIAL,OK,PARTIAL,PARTIAL,PARTIAL,PARTIAL
nqWMPgcAgXjZgEM8,OK,OK,OK,ALL MISSING,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL


### 4c. Raw Postback — Attributed vs Unattributed Revenue

Check `df_accesslog.pb` before attribution processing.  
If unattributed postbacks also show \$0 → MMP-side issue.  
If only attributed show \$0 → Moloco processing issue.

In [ ]:
if CAMPAIGN_IDS:
    camp_filter = ', '.join(f"'{c}'" for c in CAMPAIGN_IDS)
    camp_to_adv = df_campaigns.set_index('campaign_id')['advertiser_id'].to_dict()

    df_adv_pb = run_query(f"""
    SELECT
      DATE(timestamp) AS date,
      moloco.campaign_id AS campaign_id,
      attribution.attributed,
      CASE
        WHEN event.revenue_usd.amount > 0 THEN 'has_revenue'
        WHEN event.revenue_usd.amount = 0 THEN 'zero_revenue'
        ELSE 'null_revenue'
      END AS revenue_status,
      COUNT(*) AS event_count,
      ROUND(SUM(IFNULL(event.revenue_usd.amount, 0)), 4) AS total_revenue_usd
    FROM `focal-elf-631.df_accesslog.pb`
    WHERE DATE(timestamp) BETWEEN '2026-03-01' AND '2026-03-10'
      AND moloco.campaign_id IN ({camp_filter})
      AND event.name = 'revenue'
    GROUP BY 1, 2, 3, 4
    ORDER BY campaign_id, date, attributed DESC, revenue_status
    """, '4c. Raw PB (Attributed + Unattributed)')

    if not df_adv_pb.empty:
        df_adv_pb['advertiser_id'] = df_adv_pb['campaign_id'].map(camp_to_adv)

        for adv_id in ADVERTISER_IDS:
            sub = df_adv_pb[df_adv_pb['advertiser_id'] == adv_id]
            if sub.empty:
                print(f'\n{adv_id}: no raw postback revenue events found.')
                continue
            print(f'\n=== Advertiser: {adv_id} ===')

            pivot = sub.groupby(['date', 'attributed']).agg(
                events=('event_count', 'sum'),
                revenue=('total_revenue_usd', 'sum')
            ).unstack(fill_value=0)
            print('Daily revenue (attributed vs unattributed):')
            display(pivot)

            attr_sub = sub[(sub['attributed'] == True) & (sub['revenue_status'] == 'zero_revenue')]
            if not attr_sub.empty:
                print('\nAttributed zero-revenue events by campaign x date:')
                display(attr_sub[['date', 'campaign_id', 'event_count', 'total_revenue_usd']])
            else:
                print('\nNo attributed zero-revenue events found.')
    else:
        df_adv_pb = pd.DataFrame()
else:
    print('No campaigns found — skipping 4c.')
    df_adv_pb = pd.DataFrame()

✅ 4c. Raw PB (Attributed + Unattributed): 279 rows

=== Advertiser: yfg0At8VksGnt6EO ===
Daily revenue (attributed vs unattributed):


,events,revenue
attributed,True,True
date,,
2026-03-01,234,1691.9231
2026-03-02,193,1508.6016
2026-03-03,9267,21491.2328
2026-03-04,10340,26245.4287
2026-03-05,9704,30879.0229
2026-03-06,8081,24165.8752
2026-03-07,9408,29761.7853
2026-03-08,8576,26172.0673



No attributed zero-revenue events found.

=== Advertiser: tGYMnmOyEZqjEIND ===
Daily revenue (attributed vs unattributed):


,events,revenue
attributed,True,True
date,,
2026-03-01,7,68.6013
2026-03-02,4,46.8243
2026-03-03,6,52.9936
2026-03-04,8,6.2669
2026-03-05,15,14.2583
2026-03-06,17,46.9040
2026-03-07,32,42.5728
2026-03-08,35,97.7144



Attributed zero-revenue events by campaign x date:


,date,campaign_id,event_count,total_revenue_usd
40,2026-03-05,Jl5NUKiajhIyKXQK,5,0.0
41,2026-03-06,Jl5NUKiajhIyKXQK,1,0.0
43,2026-03-07,Jl5NUKiajhIyKXQK,3,0.0
45,2026-03-08,Jl5NUKiajhIyKXQK,7,0.0
47,2026-03-09,Jl5NUKiajhIyKXQK,3,0.0
49,2026-03-10,Jl5NUKiajhIyKXQK,3,0.0
167,2026-03-05,fISFGcvrsN7uXPoR,2,0.0
170,2026-03-07,fISFGcvrsN7uXPoR,5,0.0
172,2026-03-08,fISFGcvrsN7uXPoR,5,0.0
174,2026-03-09,fISFGcvrsN7uXPoR,2,0.0


### 4d. Comparison: Attributed (cv) vs Raw Postback

gap = raw\_revenue − attr\_revenue  
Large positive gap → revenue present in raw PB but lost during attribution/processing.

In [ ]:
if CAMPAIGN_IDS and not df_adv_pb.empty and not df_adv_campaign_rev.empty:
    for adv_id in ADVERTISER_IDS:
        print(f'\n=== Advertiser: {adv_id} ===')

        # Attributed (cv) daily totals
        attr = df_adv_campaign_rev[df_adv_campaign_rev['advertiser_id'] == adv_id]
        attr_daily = attr.groupby('date').agg(
            attr_events=('revenue_events', 'sum'),
            attr_with_rev=('events_with_revenue', 'sum'),
            attr_revenue=('pb_revenue', 'sum')
        ).reset_index()
        attr_daily['date'] = attr_daily['date'].astype(str)

        # Raw PB daily totals
        pb = df_adv_pb[df_adv_pb['advertiser_id'] == adv_id]
        pb_daily = pb.groupby('date').agg(
            raw_events=('event_count', 'sum'),
            raw_revenue=('total_revenue_usd', 'sum')
        ).reset_index()
        pb_daily['date'] = pb_daily['date'].astype(str)

        if attr_daily.empty and pb_daily.empty:
            print('  No data found.')
            continue

        comparison = pd.merge(attr_daily, pb_daily, on='date', how='outer').sort_values('date')
        comparison['gap'] = comparison['raw_revenue'].fillna(0) - comparison['attr_revenue'].fillna(0)
        comparison['gap_pct'] = (
            comparison['gap'] / comparison['raw_revenue'].replace(0, float('nan')) * 100
        ).round(1)

        print('Attributed (cv) vs Raw Postback comparison:')
        print('gap = raw_revenue − attr_revenue (positive = revenue dropped after PB receipt)')
        display(comparison[['date', 'attr_events', 'attr_with_rev', 'attr_revenue',
                             'raw_events', 'raw_revenue', 'gap', 'gap_pct']])
else:
    print('Insufficient data to run comparison — check 4b and 4c outputs.')


=== Advertiser: yfg0At8VksGnt6EO ===
Attributed (cv) vs Raw Postback comparison:
gap = raw_revenue − attr_revenue (positive = revenue dropped after PB receipt)


,date,attr_events,attr_with_rev,attr_revenue,raw_events,raw_revenue,gap,gap_pct
0,2026-03-01,162,162,1216.0170,234,1691.9231,475.9061,28.1
1,2026-03-02,111,111,998.1799,193,1508.6016,510.4217,33.8
2,2026-03-03,9200,9200,21238.0574,9267,21491.2328,253.1754,1.2
3,2026-03-04,10258,10258,25756.8663,10340,26245.4287,488.5624,1.9
4,2026-03-05,9401,9401,28106.8117,9704,30879.0229,2772.2112,9.0
5,2026-03-06,7927,7927,22862.8592,8081,24165.8752,1303.0160,5.4
6,2026-03-07,9258,9258,28647.3599,9408,29761.7853,1114.4254,3.7
7,2026-03-08,8504,8504,25672.4636,8576,26172.0673,499.6037,1.9
8,2026-03-09,6429,6429,24280.1942,6503,24826.6175,546.4233,2.2
9,2026-03-10,5022,5022,19828.0488,5109,20321.2049,493.1561,2.4



=== Advertiser: tGYMnmOyEZqjEIND ===
Attributed (cv) vs Raw Postback comparison:
gap = raw_revenue − attr_revenue (positive = revenue dropped after PB receipt)


,date,attr_events,attr_with_rev,attr_revenue,raw_events,raw_revenue,gap,gap_pct
0,2026-03-01,7,7,68.6013,7,68.6013,0.000000e+00,0.0
1,2026-03-02,4,4,46.8243,4,46.8243,0.000000e+00,0.0
2,2026-03-03,6,6,52.9936,6,52.9936,0.000000e+00,0.0
3,2026-03-04,8,2,6.2669,8,6.2669,0.000000e+00,0.0
4,2026-03-05,15,5,14.2583,15,14.2583,0.000000e+00,0.0
5,2026-03-06,17,10,46.9040,17,46.9040,0.000000e+00,0.0
6,2026-03-07,32,11,42.5728,32,42.5728,0.000000e+00,0.0
7,2026-03-08,35,13,97.7144,35,97.7144,-1.421085e-14,-0.0
8,2026-03-09,38,10,22.7300,38,22.7300,0.000000e+00,0.0
9,2026-03-10,33,18,154.4879,33,154.4879,0.000000e+00,0.0
